# Part 5 (Batch B) - machine translation of the English corpus (Colab, GPU)

**What this does:** translates the English **train** rows of the scam-text corpus into
Sinhala (`sin_Sinh`) and Tamil (`tam_Taml`) with Meta's NLLB-200 (`facebook/nllb-200-distilled-600M`).

**Research integrity:** machine translations are **train-only augmentation**. They never go
into validation or test, and the corpus builder drops any translation whose source row is
not in train. Do not translate anything except the exported file.

**Steps**
1. On your PC: `python scripts/build_text_corpus.py export-mt-input` → `data/corpus/mt/mt_input_en.csv`.
2. Colab: *Runtime → Change runtime type → GPU* (T4 is enough).
3. Run the cells below and upload `mt_input_en.csv` when asked.
4. Download `mt_si.csv` and `mt_ta.csv` and put them in `data/corpus/mt/` on your PC.
5. On your PC: `python scripts/build_text_corpus.py build --version v0.3` (then `stats`).
6. Spot-check some translations with `python scripts/review_corpus.py review --reviewer <name> --source-type machine_translation --random --limit 50`.

Output columns (what the builder expects): `text, language, intent, source_text_id`.

## 1. Install

In [ ]:
!pip -q install "transformers>=4.40" sentencepiece accelerate pandas

## 2. Upload `mt_input_en.csv`

In [ ]:
import pandas as pd

try:
    from google.colab import files  # only on Colab
    uploaded = files.upload()
    input_name = next(iter(uploaded))
except ImportError:
    input_name = "mt_input_en.csv"  # running locally: the file is next to the notebook

source = pd.read_csv(input_name, dtype=str, keep_default_na=False)
required = {"source_text_id", "text", "intent"}
missing = required - set(source.columns)
assert not missing, f"not the export-mt-input file, missing columns: {missing}"
source = source[source["text"].str.strip() != ""].reset_index(drop=True)
print(len(source), "English train rows to translate")
source.head()

## 3. Load NLLB-200 (distilled 600M)

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)  # CPU works but is very slow
tokenizer = AutoTokenizer.from_pretrained(MODEL, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

## 4. Translate in batches

Placeholders such as `[NAME]`, `[PHONE]`, `[EMAIL]` are protected: they are replaced by
`<0>`, `<1>`, ... before translation and put back afterwards. A row whose placeholders do
not come back is dropped, not guessed.

In [ ]:
import re

PLACEHOLDER = re.compile(r"\[[A-Z_]+\]")
BATCH_SIZE = 32
MAX_TOKENS = 256


def protect(text):
    found = PLACEHOLDER.findall(text)
    for i, token in enumerate(found):
        text = text.replace(token, f"<{i}>", 1)
    return text, found


def restore(text, found):
    for i, token in enumerate(found):
        if f"<{i}>" not in text:
            return None
        text = text.replace(f"<{i}>", token, 1)
    return text


@torch.inference_mode()
def translate(texts, target):
    out = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start : start + BATCH_SIZE]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_TOKENS
        ).to(device)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target),
            max_new_tokens=MAX_TOKENS,
            num_beams=4,
        )
        out += tokenizer.batch_decode(generated, skip_special_tokens=True)
        print(f"{target}: {min(start + BATCH_SIZE, len(texts))}/{len(texts)}", end="\r")
    print()
    return out


protected = [protect(text) for text in source["text"]]
outputs = {}
for language, code_ in [("si", "sin_Sinh"), ("ta", "tam_Taml")]:
    translated = translate([text for text, _ in protected], code_)
    rows = []
    for (_, found), text, row in zip(protected, translated, source.itertuples(), strict=True):
        text = restore(text.strip(), found)
        if text:
            rows.append({"text": text, "language": language, "intent": row.intent,
                         "source_text_id": row.source_text_id})
    outputs[language] = pd.DataFrame(rows, columns=["text", "language", "intent", "source_text_id"])
    print(language, len(outputs[language]), "rows kept of", len(source))

## 5. Check a few, then save and download

In [ ]:
for language, frame in outputs.items():
    print(language)
    for text in frame["text"].head(5):
        print("   ", text)

for language, frame in outputs.items():
    name = f"mt_{language}.csv"
    frame.to_csv(name, index=False, encoding="utf-8")
    try:
        files.download(name)
    except NameError:
        print("saved", name)

Put `mt_si.csv` and `mt_ta.csv` into `data/corpus/mt/` and rebuild the corpus. They are
git-ignored like all corpus files.